In [1]:
import pandas as pd
import json
import os
#!pip install pyyaml
import yaml
from collections import defaultdict

In [2]:
# clone repository from github at https://github.com/openstates/people.git
folder_path = "data/us/legislature"
files = [f for f in os.listdir(folder_path) if f.endswith(".yml")]
# print the files
print(files)

['Kweisi-Mfume-79575558-ef44-5bb5-9c64-3d3fe3fb4427.yml', 'Carlos-A-Gimenez-9db37a87-2ba9-56a0-9b42-89697222e044.yml', 'Christopher-H-Smith-84a22f15-cf83-5f0b-a048-a6fc50aa60fe.yml', 'Michael-R-Turner-bac6c65d-846b-5e60-9532-fa216c99ccf6.yml', 'Eric-Schmitt-999df51b-9318-55b9-b0f6-d738ffc1d62d.yml', 'John-McGuire-f41db99d-4852-4347-ae03-5be5d66f5fe4.yml', 'Hillary-J-Scholten-55404260-10e4-59e0-b228-2a01a21fb952.yml', 'Raphael-G-Warnock-37f5a4b9-98b2-5c3d-857b-6e02f5123345.yml', 'Rand-Paul-7a1c13f9-1aac-5461-aa2a-11642749828d.yml', 'Robert-J-Wittman-0adba0f5-0723-584b-8071-3e442a01f2d2.yml', 'Eugene-Vindman-4ec4f784-d347-43e9-a473-907abbd8a6c7.yml', 'Josh-Gottheimer-8ce4e2db-be5a-5493-a399-b2e6061301ba.yml', 'Mike-Crapo-9243a3a9-4e91-5fc8-9643-ae46455b688b.yml', 'Vince-Fong-2d6b7f7f-a241-5064-9812-9506beb2b76e.yml', 'Vern-Buchanan-1e646f7b-600b-503f-9524-3ae95d00cd2d.yml', 'Sydney-Kamlager-Dove-1608d897-8cf1-5902-9e45-fbd0bbd72cca.yml', 'Bill-Cassidy-851546e4-4af0-5505-8a1c-4f2a83813589

In [3]:
# Convert all the yaml files to a dataframe
rows = []

# First pass: gather all possible identifier schemes
all_schemes = set()

for file in files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
        for id_obj in data.get("other_identifiers", []):
            all_schemes.add(id_obj.get("scheme"))

# Now process each file with all identifier columns
for file in files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

        # Prepare default row
        row = defaultdict(lambda: None)

        # Basic fields
        row["id"] = data.get("id")
        row["name"] = data.get("name")
        row["given_name"] = data.get("given_name")
        row["family_name"] = data.get("family_name")
        row["birth_date"] = data.get("birth_date")
        row["gender"] = data.get("gender")
        row["email"] = data.get("email")
        row["image"] = data.get("image")
        row["party"] = data.get("party", [{}])[0].get("name")

        # Roles
        role = data.get("roles", [{}])[-1]
        row["role_type"] = role.get("type")
        row["district"] = role.get("district")
        row["role_start_date"] = role.get("start_date")
        row["role_end_date"] = role.get("end_date")

        # Social media (if available)
        for k, v in data.get("ids", {}).items():
            row[f"social_{k}"] = v

        # Other identifiers
        for id_obj in data.get("other_identifiers", []):
            scheme = id_obj.get("scheme")
            identifier = id_obj.get("identifier")
            row[f"{scheme}_id"] = identifier

        rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

# Fill in missing columns for any scheme not found in every legislator
for scheme in all_schemes:
    col_name = f"{scheme}_id"
    if col_name not in df.columns:
        df[col_name] = None

In [4]:
df.columns

Index(['id', 'name', 'given_name', 'family_name', 'birth_date', 'gender',
       'email', 'image', 'party', 'role_type', 'district', 'role_start_date',
       'role_end_date', 'social_twitter', 'social_facebook', 'ballotpedia_id',
       'bioguide_id', 'fec_id', 'google_entity_id_id', 'govtrack_id',
       'house_history_id', 'icpsr_id', 'opensecrets_id', 'pictorial_id',
       'thomas_id', 'votesmart_id', 'wikidata_id', 'wikipedia_id',
       'social_youtube', 'cspan_id', 'maplight_id', 'lis_id'],
      dtype='object')

In [5]:
df.head()

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,opensecrets_id,pictorial_id,thomas_id,votesmart_id,wikidata_id,wikipedia_id,social_youtube,cspan_id,maplight_id,lis_id
0,ocd-person/79575558-ef44-5bb5-9c64-3d3fe3fb4427,Kweisi Mfume,Kweisi,Mfume,1948-10-24,Male,https://mfume.house.gov/address_authentication...,https://unitedstates.github.io/images/congress...,Democratic,lower,...,N00001799,13090,00798,26892,Q519504,NaN,NaN,NaN,NaN,NaN
1,ocd-person/9db37a87-2ba9-56a0-9b42-89697222e044,Carlos Giménez,Carlos,Giménez,1954-01-17,Male,https://gimenez.house.gov/contact,https://unitedstates.github.io/images/congress...,Republican,lower,...,N00046394,13009,NaN,81366,Q5041653,Carlos A. Giménez,NaN,NaN,NaN,NaN
2,ocd-person/84a22f15-cf83-5f0b-a048-a6fc50aa60fe,Chris Smith,Chris,Smith,1953-03-04,Male,https://chrissmith.house.gov/contact/zipauth.htm,https://unitedstates.github.io/images/congress...,Republican,lower,...,N00009816,13153,01071,26952,Q981167,Chris Smith (New Jersey politician),UCtCNUDo3-I1gsd_03ppDfZg,6411,469,NaN
3,ocd-person/bac6c65d-846b-5e60-9532-fa216c99ccf6,Mike Turner,Mike,Turner,1960-01-11,Male,https://turner.house.gov/email-me,https://unitedstates.github.io/images/congress...,Republican,lower,...,N00025175,13204,01741,45519,Q505722,Mike Turner,UC-6Ss-aZ3OPisf9GdVGtN8g,1003607,496,NaN
4,ocd-person/999df51b-9318-55b9-b0f6-d738ffc1d62d,Eric Schmitt,Eric,Schmitt,1975-06-20,Male,https://www.schmitt.senate.gov/contact/,https://unitedstates.github.io/images/congress...,Republican,upper,...,N00048414,13416,NaN,104474,Q5387455,NaN,NaN,NaN,NaN,S420


In [6]:
df[["id", "name", "bioguide_id"]]

,id,name,bioguide_id
0,ocd-person/79575558-ef44-5bb5-9c64-3d3fe3fb4427,Kweisi Mfume,M000687
1,ocd-person/9db37a87-2ba9-56a0-9b42-89697222e044,Carlos Giménez,G000593
2,ocd-person/84a22f15-cf83-5f0b-a048-a6fc50aa60fe,Chris Smith,S000522
3,ocd-person/bac6c65d-846b-5e60-9532-fa216c99ccf6,Mike Turner,T000463
4,ocd-person/999df51b-9318-55b9-b0f6-d738ffc1d62d,Eric Schmitt,S001227
...,...,...,...
533,ocd-person/86b65fd7-4549-51fa-80e8-eaf3daf3e60e,Lou Correa,C001110
534,ocd-person/10de7024-4b40-57b0-ae78-101372fd4a02,Pat Ryan,R000579
535,ocd-person/39f36070-b860-5345-a3cc-ea7fdbf7dfb3,Adrian Smith,S001172
536,ocd-person/74d8d6c8-c349-4fe7-ae18-62c69c4f8d4b,Craig Goldman,G000601


In [10]:
# are there any missing bioguide ids?
df[df['bioguide_id'].isna()].head()

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,opensecrets_id,pictorial_id,thomas_id,votesmart_id,wikidata_id,wikipedia_id,social_youtube,cspan_id,maplight_id,lis_id


In [11]:
# Save file to csv
df.to_csv("plural_legislators_with_bioguide.csv", index=False)